# Sentiment Analysis Project

This notebook demonstrates the complete workflow for building a sentiment analysis model using machine learning. We'll work with the IMDB movie review dataset to classify text as positive or negative sentiment.

## Overview

The project involves:
1. **Data Loading**: Importing and exploring the IMDB dataset
2. **Text Preprocessing**: Cleaning and preparing text for analysis
3. **Feature Extraction**: Converting text to numerical features using TF-IDF
4. **Model Training**: Training and evaluating machine learning models
5. **Model Deployment**: Saving the trained model for use in a web application

## Requirements

Before running this notebook, ensure you have the following libraries installed:
- pandas
- nltk
- scikit-learn
- numpy
- scipy

You'll also need to download NLTK data packages for text processing.

# Step 1: Data Loading and Exploration

First, we need to load our dataset. We're using the IMDB Dataset which contains 50,000 movie reviews labeled as positive or negative.

The dataset is stored in a CSV file with columns for the review text and sentiment labels. We'll use pandas to read the data and take a quick look at its structure.

In [24]:
# preprocessing.py
import pandas as pd

# Load the CSV with proper separator and encoding
df = pd.read_csv("IMDB Dataset.csv")

print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [3]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from nltk import pos_tag

# Download NLTK data if not already
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

# Initialize Lemmatizer
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gauta\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gauta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gauta\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\gauta\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gauta\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


# Step 2: Setting up NLTK and Text Processing Tools

Natural Language Toolkit (NLTK) is a powerful library for working with human language data. We'll need several NLTK components for text preprocessing:

- **punkt**: For sentence tokenization
- **stopwords**: Common words to filter out
- **wordnet**: Lexical database for lemmatization
- **omw-1.4**: Open Multilingual Wordnet
- **averaged_perceptron_tagger**: For part-of-speech tagging

These downloads are required for the preprocessing steps we'll implement later.

In [4]:
lemmatizer = WordNetLemmatizer()

# Step 3: Initializing Text Processing Components

Before we can preprocess text, we need to set up our core processing tools:

- **WordNetLemmatizer**: Reduces words to their base or dictionary form (e.g., "running" → "run")
- **Stopwords**: Common words like "the", "is", "and" that don't carry much meaning
- **POS Tagging Functions**: To determine word types for better lemmatization

These components will be used in our custom preprocessing pipeline.

In [5]:
def get_wordnet_pos(treebank_tag):
    """Map POS tag to WordNet POS"""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

# Step 4: Defining Helper Functions for Text Processing

We need several utility functions for advanced text preprocessing:

- **get_wordnet_pos()**: Maps NLTK's POS tags to WordNet's format for better lemmatization
- **lemmatize_tokens()**: Applies lemmatization to a list of tokens
- **advanced_negation()**: Handles negation words by prefixing subsequent words (e.g., "not good" becomes "not_good")

These functions will be used within our scikit-learn compatible transformers.

In [6]:
def lemmatize_tokens(tokens):
    pos_tags = pos_tag(tokens)
    return [lemmatizer.lemmatize(word, get_wordnet_pos(pos)) for word, pos in pos_tags]

# Step 5: Advanced Text Preprocessing Functions

The `advanced_negation()` function implements a sophisticated approach to handling negation in text. Instead of simply removing negation words, it prefixes the following words with "not_" until a punctuation mark is encountered.

For example:
- Input: "I don't like this movie"
- Output: ["I", "not_like", "not_this", "not_movie"]

This preserves the semantic impact of negation on the sentiment of the text.

In [7]:
import re

def advanced_negation(tokens):
    negation_words = {
        "not", "no", "never", "none", "nobody", "nothing",
        "dont", "don't", "didnt", "didn't", "doesnt", "doesn't",
        "isnt", "isn't", "wasnt", "wasn't",
        "shouldnt", "shouldn't", "cant", "can't", "cannot"
    }

    punctuation = {".", ",", "!", "?", ";", ":"}
    
    new_tokens = []
    negate = False

    for word in tokens:

        # Stop negation at punctuation
        if word in punctuation:
            negate = False
            new_tokens.append(word)
            continue

        # If current word is a negation word
        if word.lower() in negation_words:
            negate = True
            continue  # Don't add the negation word itself

        # If negation is active → prefix all words
        if negate:
            new_tokens.append("not_" + word)
        else:
            new_tokens.append(word)

    return new_tokens

# Step 6: Implementing the Negation Handler

The `advanced_negation()` function processes a list of tokens and handles negation by:

1. Detecting common negation words ("not", "don't", "never", etc.)
2. When negation is active, prefixing subsequent words with "not_"
3. Stopping negation at punctuation marks

This approach helps capture the nuanced meaning of negated phrases in sentiment analysis.

# Text Preprocessing Pipeline

This section defines a custom preprocessing pipeline using scikit-learn's `Pipeline` and custom transformers. Each step in the pipeline is implemented as a class that inherits from `BaseEstimator` and `TransformerMixin`, allowing it to be used seamlessly in scikit-learn pipelines.

The pipeline consists of the following steps:

1. **HTMLCleaner**: Removes HTML tags from text
2. **TextNormalizer**: Converts text to lowercase and removes special characters
3. **Tokenizer**: Splits text into individual words (tokens)
4. **NegationHandler**: Handles negation words by prefixing subsequent words with "not_"
5. **StopwordRemover**: Removes common stopwords (e.g., "the", "is", "and")
6. **Lemmatizer**: Reduces words to their base or root form using WordNet lemmatizer
7. **TokenJoiner**: Joins tokens back into a single string
8. **RareWordRemover**: Removes words that appear less than a minimum frequency threshold

This pipeline transforms raw text into clean, processed text suitable for feature extraction and machine learning models.

In [10]:
import re
from collections import Counter
from nltk.tokenize import word_tokenize
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
# Step 1: HTML Tag Remover
class HTMLCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X, y=None):
        return [re.sub(r'<.*?>', '', text) for text in X]

# Step 2: Lowercasing + Special Char Removal
class TextNormalizer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X, y=None):
        result = []
        for text in X:
            text = text.lower()
            text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
            text = re.sub(r'\s+', ' ', text).strip()
            result.append(text)
        return result

# Step 3: Tokenizer
class Tokenizer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X, y=None):
        return [word_tokenize(text) for text in X]

# Step 4: Negation Handler
class NegationHandler(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X, y=None):
        return [advanced_negation(tokens) for tokens in X]

# Step 5: Stopword Remover
class StopwordRemover(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X, y=None):
        return [[w for w in tokens if w not in stop_words] for tokens in X]

# Step 6: Lemmatizer
class Lemmatizer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X, y=None):
        return [lemmatize_tokens(tokens) for tokens in X]

# Step 7: Token Joiner (tokens → string)
class TokenJoiner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X, y=None):
        return [' '.join(tokens) for tokens in X]

# Step 8: Rare Word Remover
class RareWordRemover(BaseEstimator, TransformerMixin):
    def __init__(self, min_freq=2): self.min_freq = min_freq
    def fit(self, X, y=None):
        all_tokens = ' '.join(X).split()
        counts = Counter(all_tokens)
        self.rare_words_ = {w for w, c in counts.items() if c < self.min_freq}
        return self
    def transform(self, X, y=None):
        return [' '.join(w for w in text.split() if w not in self.rare_words_) for text in X]

# Creating the Preprocessing Pipeline

Now that we have defined all the custom transformer classes, we can create a scikit-learn `Pipeline` that chains them together. The pipeline allows us to apply all preprocessing steps in sequence with a single `fit_transform()` call.

The pipeline is defined as a list of tuples, where each tuple contains:
- A string name for the step
- An instance of the transformer class

The order of steps is important and follows the logical flow of text preprocessing.

In [ ]:
text_preprocessing_pipeline = Pipeline([
    ('html_cleaner',      HTMLCleaner()),
    ('normalizer',        TextNormalizer()),
    ('tokenizer',         Tokenizer()),
    ('negation_handler',  NegationHandler()),
    ('stopword_remover',  StopwordRemover()),
    ('lemmatizer',        Lemmatizer()),
    ('rare_word_remover', RareWordRemover(min_freq=2)),
    ('token_joiner',      TokenJoiner())
])

# Applying the Preprocessing Pipeline

With the pipeline created, we can now apply it to our dataset. The `fit_transform()` method first fits the pipeline (learning parameters like rare words from the training data) and then transforms the text data.

This creates a new column 'clean_review' in our dataframe containing the preprocessed text, ready for feature extraction.

In [12]:
df['clean_review'] = text_preprocessing_pipeline.fit_transform(df['review'])

In [13]:
df['clean_review'].head(3)

0    one reviewer mention watch 1 oz episode youll ...
1    wonderful little production film technique una...
2    think wonderful way spend time hot summer week...
Name: clean_review, dtype: object

# Preparing Labels for Machine Learning

Before training our model, we need to convert the categorical sentiment labels ('positive'/'negative') into numerical values that machine learning algorithms can work with. We'll map:
- 'positive' → 1
- 'negative' → 0

This binary encoding is suitable for classification tasks.

In [15]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Splitting Data into Training and Testing Sets

To evaluate our model's performance properly, we need to split our data into training and testing sets. This prevents overfitting and allows us to assess how well the model generalizes to unseen data.

We'll use 80% of the data for training and 20% for testing, with stratification to maintain the same class distribution in both sets.

In [16]:
from sklearn.model_selection import train_test_split

X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y  # ensures same class distribution in train & test
)

# Feature Extraction with TF-IDF

Now we need to convert our preprocessed text into numerical features that machine learning algorithms can understand. We'll use TF-IDF (Term Frequency-Inverse Document Frequency) vectorization, which:

- **Term Frequency (TF)**: Measures how frequently a term appears in a document
- **Inverse Document Frequency (IDF)**: Reduces the weight of commonly used words
- **TF-IDF**: Balances these to give higher weight to terms that are frequent in a document but rare across the corpus

We're using up to 50,000 features, including unigrams and bigrams, and removing English stopwords.

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50000, stop_words='english', ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Model Training and Evaluation

With our features ready, we can now train a machine learning model. We'll start with Multinomial Naive Bayes, which is well-suited for text classification tasks with TF-IDF features.

Naive Bayes classifiers work by applying Bayes' theorem with the assumption that features are independent. Multinomial NB is specifically designed for discrete features like word counts.

In [18]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
preds = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

Accuracy: 0.8132
              precision    recall  f1-score   support

           0       0.81      0.81      0.81      5000
           1       0.81      0.81      0.81      5000

    accuracy                           0.81     10000
   macro avg       0.81      0.81      0.81     10000
weighted avg       0.81      0.81      0.81     10000



# Hyperparameter Tuning with Logistic Regression

While Naive Bayes gave us a baseline, we can improve performance with Logistic Regression, which is more flexible. We'll use RandomizedSearchCV to find optimal hyperparameters:

- **C**: Regularization strength (inverse of lambda)
- **solver**: Optimization algorithm

Randomized search samples from the parameter space randomly, which is more efficient than grid search for large parameter spaces.

In [19]:

from sklearn.linear_model import LogisticRegression
from scipy.stats import uniform
from sklearn.model_selection import RandomizedSearchCV

# Search space
log_reg = LogisticRegression(
    max_iter=1000,
    penalty='l2'
)

param_dist = {
    'C': uniform(0.01, 100),
    'solver': ['liblinear', 'lbfgs']
}

random_search = RandomizedSearchCV(
    log_reg,
    param_dist,
    n_iter=20,
    cv=5,
    n_jobs=-1
)
# Train
random_search.fit(X_train_tfidf, y_train)

print("Best Params:", random_search.best_params_)
print("Best Score:", random_search.best_score_)
lr_preds = random_search.predict(X_test_tfidf)
print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, lr_preds))
print(classification_report(y_test, lr_preds))

Best Params: {'C': np.float64(6.93141798041802), 'solver': 'liblinear'}
Best Score: 0.8929
=== Logistic Regression ===
Accuracy: 0.898
              precision    recall  f1-score   support

           0       0.91      0.89      0.90      5000
           1       0.89      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [20]:
best_lr_model = random_search.best_estimator_

# Saving the Trained Model and Components

To use our trained model in the Flask web application, we need to save:
1. The preprocessing pipeline
2. The TF-IDF vectorizer
3. The trained sentiment model

We'll use pickle to serialize these Python objects so they can be loaded and used for predictions on new text.

In [22]:
import pickle

# Save TF-IDF vectorizer
with open("tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

# Save model (NB or Logistic)
with open("sentiment_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Files saved: tfidf.pkl, sentiment_model.pkl")

Files saved: tfidf.pkl, sentiment_model.pkl


In [23]:


with open("text_preprocessing_pipeline.pkl", "wb") as f:
    pickle.dump(text_preprocessing_pipeline, f)